# Baltic Sea Current Profile Explorer

Interactive 3D visualisation of the full-depth current profile at a single geographic point.

**What it shows:** At each depth level a cone arrow represents the horizontal ocean current
(eastward `uo`, northward `vo`). The Z-axis is depth (0 at surface, ~80 m at bottom).
Use the time slider or Play button to watch the vertical current structure evolve.

**Usage:** Edit the **Parameters** cell below, then run all cells (`Kernel › Restart & Run All`).

**No interpolation is performed** — the nearest grid cell to the requested lat/lon is used directly.

In [1]:
# ── USER PARAMETERS ──────────────────────────────────────────────────────────
# Edit these values, then run all cells.

LAT        = 55.05                    # decimal degrees N
LON        = 14.3                   # decimal degrees E
START_TIME = "2023-10-01T00:00:00"   # ISO-8601, inclusive
STOP_TIME  = "2024-10-03T00:00:00"   # ISO-8601, inclusive

# Max animation frames. If the time window has more hourly steps than this,
# every Kth step is kept automatically to keep the browser responsive.
N_FRAMES_MAX = 2000

In [2]:
import sys
import math
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import plotly.graph_objects as go

sys.path.insert(0, str(Path("..") / "src"))
from data_loader import load_manifest, select_tiles, load_working_window

DATA_DIR = Path("..") / "data" / "raw"

In [3]:
start_dt = datetime.fromisoformat(START_TIME)
stop_dt  = datetime.fromisoformat(STOP_TIME)

manifest = load_manifest(DATA_DIR)
ds_lazy  = select_tiles(manifest, DATA_DIR, LAT, LON, start_dt, stop_dt)

# spatial_margin_deg=0.1 keeps only nearest grid cells, saving ~30x memory vs the 1.0 default
ds_window = load_working_window(
    ds_lazy, LAT, LON, start_dt, stop_dt, spatial_margin_deg=0.1
)

# Nearest grid cell — no interpolation
ds_pt = ds_window.sel(latitude=LAT, longitude=LON, method="nearest")

actual_lat = float(ds_pt.latitude)
actual_lon = float(ds_pt.longitude)
print(f"Target point :  lat={LAT:.4f}°N, lon={LON:.4f}°E")
print(f"Nearest cell :  lat={actual_lat:.4f}°N, lon={actual_lon:.4f}°E")
print(f"Grid offset  :  Δlat={actual_lat - LAT:+.4f}°, Δlon={actual_lon - LON:+.4f}°")
print(f"Time steps   :  {len(ds_pt.time)} hourly steps ({start_dt} → {stop_dt})")
print(f"Depth levels :  {len(ds_pt.depth)} "
      f"({float(ds_pt.depth[0]):.2f} m → {float(ds_pt.depth[-1]):.2f} m)")

/home/ddyob/Documents/argo_piloting/sim/.venv/bin/python: No module named pip


Target point :  lat=55.0500°N, lon=14.3000°E
Nearest cell :  lat=55.0583°N, lon=14.2916°E
Grid offset  :  Δlat=+0.0083°, Δlon=-0.0084°
Time steps   :  8833 hourly steps (2023-10-01 00:00:00 → 2024-10-03 00:00:00)
Depth levels :  26 (0.50 m → 80.07 m)


In [4]:
depths = ds_pt.depth.values.astype(np.float64)   # (26,) metres, positive down
times  = ds_pt.time.values                        # (N_t,) datetime64
uo_all = ds_pt["uo"].values.astype(np.float64)   # (N_t, 26) m/s
vo_all = ds_pt["vo"].values.astype(np.float64)   # (N_t, 26) m/s

assert not np.all(np.isnan(uo_all)), (
    "Selected point is fully masked — choose a location in open water, away from the coast."
)

# Auto-stride so animation stays within N_FRAMES_MAX
n_total = len(times)
stride  = max(1, math.ceil(n_total / N_FRAMES_MAX))
idx     = np.arange(0, n_total, stride)

uo_sub = uo_all[idx]   # (N_frames, 26)
vo_sub = vo_all[idx]
t_sub  = times[idx]
n_frames = len(idx)

# Fixed colorscale across all frames (speed in m/s)
speed_all = np.sqrt(uo_sub**2 + vo_sub**2)
speed_max = max(float(np.nanmax(speed_all)), 0.01)
CMIN, CMAX = 0.0, speed_max * 1.1

# Symmetric velocity axis range for X and Y
uv_abs_max = float(np.nanmax(np.abs(np.concatenate([uo_sub.ravel(), vo_sub.ravel()]))))
uv_max = max(uv_abs_max * 1.2, 0.05)

time_labels = [pd.Timestamp(t).strftime("%Y-%m-%d %H:%M") for t in t_sub]
frame_names = [str(i) for i in range(n_frames)]

print(f"Total time steps : {n_total}")
print(f"Stride           : {stride}h")
print(f"Animation frames : {n_frames}")
print(f"Speed range      : {CMIN:.4f} – {CMAX:.4f} m/s")

Total time steps : 8833
Stride           : 5h
Animation frames : 1767
Speed range      : 0.0000 – 0.4278 m/s


In [5]:
def make_cone_trace(i: int, show_colorbar: bool = False) -> go.Cone:
    u = uo_sub[i]
    v = vo_sub[i]
    w = np.zeros_like(u)
    speed = np.sqrt(u**2 + v**2)
    return go.Cone(
        x=np.zeros(len(depths)),
        y=np.zeros(len(depths)),
        z=depths,
        u=u, v=v, w=w,
        sizemode="absolute",
        sizeref=15,
        anchor="tail",
        colorscale="Viridis",
        cmin=CMIN, cmax=CMAX, cauto=False,
        showscale=show_colorbar,
        colorbar=dict(
            title=dict(text="Speed (m/s)", side="right"),
            thickness=15, len=0.6,
        ) if show_colorbar else None,
        customdata=speed,
        hovertemplate=(
            "Depth: %{z:.1f} m<br>"
            "u (east): %{u:.4f} m/s<br>"
            "v (north): %{v:.4f} m/s<br>"
            "Speed: %{customdata:.4f} m/s"
            "<extra></extra>"
        ),
        showlegend=False,
        name="Current vectors",
    )


spine = go.Scatter3d(
    x=np.zeros(len(depths)),
    y=np.zeros(len(depths)),
    z=depths,
    mode="lines+markers",
    line=dict(color="rgba(200,200,200,0.4)", width=2),
    marker=dict(size=2, color="rgba(200,200,200,0.5)"),
    hovertemplate="Depth: %{z:.1f} m<extra>Depth column</extra>",
    showlegend=False,
    name="Depth column",
)

frames = [
    go.Frame(
        data=[make_cone_trace(i, show_colorbar=False)],
        name=frame_names[i],
        traces=[0],   # update only the cone trace; spine (trace 1) is static
    )
    for i in range(n_frames)
]

print(f"Built {n_frames} frames × {len(depths)} depth levels")

Built 1767 frames × 26 depth levels


In [6]:
slider_steps = [
    dict(
        args=[
            [frame_names[i]],
            dict(frame=dict(duration=0, redraw=True), mode="immediate",
                 transition=dict(duration=0)),
        ],
        label=time_labels[i],
        method="animate",
    )
    for i in range(n_frames)
]

fig = go.Figure(
    data=[
        make_cone_trace(0, show_colorbar=True),   # trace 0 — animated
        spine,                                     # trace 1 — static
    ],
    frames=frames,
)

fig.update_layout(
    title=dict(
        text=(
            f"Current Profile — lat={actual_lat:.4f}°N, lon={actual_lon:.4f}°E<br>"
            f"<sup>{time_labels[0]} → {time_labels[-1]}"
            f"  |  stride={stride}h  |  {n_frames} frames</sup>"
        ),
        x=0.5, xanchor="center",
        font=dict(size=14),
    ),
    scene=dict(
        xaxis=dict(
            title="Eastward velocity (m/s)",
            range=[-uv_max/10, uv_max/10],
            gridcolor="grey",
            zeroline=True, zerolinecolor="white", zerolinewidth=2,
        ),
        yaxis=dict(
            title="Northward velocity (m/s)",
            range=[-uv_max/10, uv_max/10],
            gridcolor="grey",
            zeroline=True, zerolinecolor="white", zerolinewidth=2,
        ),
        zaxis=dict(
            title="Depth (m)",
            autorange="reversed",   # 0 m at top
            gridcolor="grey",
        ),
        bgcolor="rgb(10,30,60)",
        camera=dict(eye=dict(x=1.5, y=1.5, z=0.8)),
        aspectmode="manual",
        aspectratio=dict(x=1, y=1, z=2),
    ),
    paper_bgcolor="rgb(15,20,40)",
    font_color="white",
    height=750,
    margin=dict(l=0, r=0, t=80, b=130),
    updatemenus=[
        dict(
            type="buttons",
            showactive=False,
            x=0.0, xanchor="left",
            y=-0.05, yanchor="top",
            font=dict(color="black"),
            buttons=[
                dict(
                    label="Play",
                    method="animate",
                    args=[None, dict(
                        frame=dict(duration=300, redraw=True),
                        fromcurrent=True,
                        transition=dict(duration=0),
                    )],
                ),
                dict(
                    label="Pause",
                    method="animate",
                    args=[[None], dict(
                        frame=dict(duration=0, redraw=False),
                        mode="immediate",
                        transition=dict(duration=0),
                    )],
                ),
            ],
        )
    ],
    sliders=[
        dict(
            active=0,
            steps=slider_steps,
            x=0.0, y=-0.08,
            len=1.0,
            xanchor="left", yanchor="top",
            pad=dict(b=10, t=50),
            currentvalue=dict(
                prefix="Time: ",
                visible=True,
                xanchor="center",
                font=dict(size=12, color="white"),
            ),
            tickcolor="white",
            font=dict(color="white"),
        )
    ],
)

fig.show()

## Reading the plot

| Element | Meaning |
|---|---|
| Arrow direction in XY plane | Ocean current direction (X = east, Y = north) |
| Arrow length / colour | Current speed (Viridis: dark = slow, yellow = fast) |
| Arrow position on Z axis | Depth of that measurement |
| Z axis top → bottom | Sea surface (0 m) → near-bottom (~80 m) |
| Grey spine | Vertical water column at the selected grid cell |
| Missing arrows | No data at that depth (seabed mask or NaN) |

**Navigation**
- Drag the **time slider** to jump to any timestep.
- Press **Play** to animate. **Pause** to freeze.
- Click and drag the 3D scene to rotate the view.
- Scroll to zoom.

## Surface Current Autocorrelation

How persistent are the surface currents at this location? If the current is
northeastward now, how likely is it to still be northeastward in 6h, 12h, 24h, 48h?

This tells us the **memory timescale** of the flow — and by proxy, the timescale
over which a model-vs-reality correction (from a GPS fix at surfacing) remains
useful for forecasting the next dive.

We compute the autocorrelation of the surface (shallowest depth level) `uo` and
`vo` time series at lags from 0 to 7 days, and estimate the e-folding
decorrelation timescale τ.

In [7]:
# ── Surface current autocorrelation ─────────────────────────────────────────
# Uses uo_all, vo_all already loaded above (shape: N_t × N_depth, hourly).
# Depth index 0 = shallowest level (0.50 m).

u_surface = uo_all[:, 0].copy()  # (N_t,)
v_surface = vo_all[:, 0].copy()

# Fill isolated NaNs with linear interpolation so autocorr doesn't break
u_series = pd.Series(u_surface).interpolate(limit=5)
v_series = pd.Series(v_surface).interpolate(limit=5)

# Compute autocorrelation at lags 0 to 168 hours (7 days)
max_lag_hours = 168
lags = np.arange(0, max_lag_hours + 1)

acf_u = np.array([u_series.autocorr(lag=l) for l in lags])
acf_v = np.array([v_series.autocorr(lag=l) for l in lags])

# Also compute autocorrelation of the current *speed* and *direction*
speed_series = np.sqrt(u_series**2 + v_series**2)
direction_series = np.arctan2(v_series, u_series)  # radians

acf_speed = np.array([speed_series.autocorr(lag=l) for l in lags])

# For direction, use circular autocorrelation: mean of cos(θ(t) - θ(t+lag))
dir_vals = direction_series.values
acf_dir = np.full(len(lags), np.nan)
for i, lag in enumerate(lags):
    if lag == 0:
        acf_dir[i] = 1.0
        continue
    d1 = dir_vals[:-lag]
    d2 = dir_vals[lag:]
    valid = ~np.isnan(d1) & ~np.isnan(d2)
    if valid.sum() > 100:
        acf_dir[i] = np.mean(np.cos(d1[valid] - d2[valid]))

print(f"Surface depth: {depths[0]:.2f} m")
print(f"Time series length: {len(u_series)} hours ({len(u_series)/24:.0f} days)")
print(f"NaN fraction: u={u_series.isna().mean():.1%}, v={v_series.isna().mean():.1%}")

Surface depth: 0.50 m
Time series length: 8833 hours (368 days)
NaN fraction: u=0.0%, v=0.0%


In [11]:
# ── Plot the correlogram ────────────────────────────────────────────────────
from plotly.subplots import make_subplots
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=(
        "Autocorrelation of Surface Current Components",
        "Autocorrelation of Current Speed & Direction",
    ),
    vertical_spacing=0.12,
)

fig.add_trace(go.Scatter(x=lags, y=acf_u, name="u (eastward)", line=dict(color="#1f77b4")), row=1, col=1)
fig.add_trace(go.Scatter(x=lags, y=acf_v, name="v (northward)", line=dict(color="#ff7f0e")), row=1, col=1)
fig.add_trace(go.Scatter(x=lags, y=acf_speed, name="speed", line=dict(color="#2ca02c")), row=2, col=1)
fig.add_trace(go.Scatter(x=lags, y=acf_dir, name="direction (circular)", line=dict(color="#d62728")), row=2, col=1)

# Reference lines
for row in [1, 2]:
    fig.add_hline(y=1/np.e, line_dash="dash", line_color="grey",
                  annotation_text="1/e (decorrelation threshold)",
                  annotation_position="top right", row=row, col=1)
    fig.add_hline(y=0, line_dash="dot", line_color="lightgrey", row=row, col=1)

fig.update_xaxes(title_text="Lag (hours)", row=2, col=1)
fig.update_xaxes(title_text="Lag (hours)", row=1, col=1)
fig.update_yaxes(title_text="Autocorrelation", row=1, col=1)
fig.update_yaxes(title_text="Autocorrelation", row=2, col=1)

fig.update_layout(
    height=650,
    template="plotly_white",
    title_text=f"Surface Current Autocorrelation — lat={LAT}, lon={LON}",
    showlegend=True,
)
fig.show()

In [12]:
fig.show()

In [9]:
# ── Estimate decorrelation timescale τ ─────────────────────────────────────
# τ = lag at which autocorrelation first drops below 1/e ≈ 0.368
# This is the e-folding time of an Ornstein-Uhlenbeck process.

threshold = 1.0 / np.e

def estimate_tau(acf, lags, label):
    """Find first lag where ACF drops below 1/e."""
    valid = ~np.isnan(acf)
    below = np.where(valid & (acf < threshold))[0]
    if len(below) == 0:
        print(f"  {label}: ACF never drops below 1/e within {lags[-1]}h — τ > {lags[-1]}h")
        return None
    first = below[0]
    # Linear interpolation between the two bounding lags
    if first > 0 and valid[first - 1]:
        acf_hi = acf[first - 1]
        acf_lo = acf[first]
        frac = (acf_hi - threshold) / (acf_hi - acf_lo)
        tau = lags[first - 1] + frac * (lags[first] - lags[first - 1])
    else:
        tau = float(lags[first])
    print(f"  {label}: τ ≈ {tau:.1f} hours ({tau/24:.1f} days)")
    return tau

print("Decorrelation timescales (1/e crossing):")
tau_u = estimate_tau(acf_u, lags, "u (eastward)")
tau_v = estimate_tau(acf_v, lags, "v (northward)")
tau_speed = estimate_tau(acf_speed, lags, "speed")
tau_dir = estimate_tau(acf_dir, lags, "direction")

print()
taus = [t for t in [tau_u, tau_v] if t is not None]
if taus:
    tau_mean = np.mean(taus)
    print(f"Mean component τ ≈ {tau_mean:.0f} hours ({tau_mean/24:.1f} days)")
    print()
    print("Interpretation for Kalman filter design:")
    print(f"  - A correction from a GPS fix is useful for roughly {tau_mean:.0f}h")
    print(f"  - Cycles shorter than ~{tau_mean:.0f}h can benefit from bias correction")
    print(f"  - Cycles longer than ~{2*tau_mean:.0f}h: the correction has mostly decorrelated")
    print(f"  - For the KF process model: use τ ≈ {tau_mean:.0f}h in the OU decay term")

Decorrelation timescales (1/e crossing):
  u (eastward): τ ≈ 18.8 hours (0.8 days)
  v (northward): τ ≈ 20.0 hours (0.8 days)
  speed: τ ≈ 6.7 hours (0.3 days)
  direction: τ ≈ 19.0 hours (0.8 days)

Mean component τ ≈ 19 hours (0.8 days)

Interpretation for Kalman filter design:
  - A correction from a GPS fix is useful for roughly 19h
  - Cycles shorter than ~19h can benefit from bias correction
  - Cycles longer than ~39h: the correction has mostly decorrelated
  - For the KF process model: use τ ≈ 19h in the OU decay term


In [10]:
# ── Seasonal breakdown ──────────────────────────────────────────────────────
# The decorrelation timescale likely varies by season. Storms in winter
# may cause faster-changing currents than calm summer conditions.

time_index = pd.DatetimeIndex(times)
months = time_index.month

seasons = {
    "DJF (winter)": [12, 1, 2],
    "MAM (spring)": [3, 4, 5],
    "JJA (summer)": [6, 7, 8],
    "SON (autumn)": [9, 10, 11],
}

fig_seasonal = make_subplots(
    rows=2, cols=2,
    subplot_titles=list(seasons.keys()),
    vertical_spacing=0.12,
    horizontal_spacing=0.08,
)

seasonal_taus = {}
max_lag_seasonal = 120  # 5 days — enough to see the shape
lags_s = np.arange(0, max_lag_seasonal + 1)

for i, (season_name, month_list) in enumerate(seasons.items()):
    row, col = divmod(i, 2)
    row += 1
    col += 1

    mask = np.isin(months, month_list)
    if mask.sum() < max_lag_seasonal * 2:
        print(f"{season_name}: not enough data ({mask.sum()} hours), skipping")
        continue

    u_s = pd.Series(u_surface[mask]).interpolate(limit=5)
    v_s = pd.Series(v_surface[mask]).interpolate(limit=5)

    acf_u_s = np.array([u_s.autocorr(lag=l) for l in lags_s])
    acf_v_s = np.array([v_s.autocorr(lag=l) for l in lags_s])

    fig_seasonal.add_trace(
        go.Scatter(x=lags_s, y=acf_u_s, name="u", line=dict(color="#1f77b4"),
                   showlegend=(i == 0)),
        row=row, col=col,
    )
    fig_seasonal.add_trace(
        go.Scatter(x=lags_s, y=acf_v_s, name="v", line=dict(color="#ff7f0e"),
                   showlegend=(i == 0)),
        row=row, col=col,
    )
    fig_seasonal.add_hline(y=1/np.e, line_dash="dash", line_color="grey",
                           row=row, col=col)

    # Estimate τ for this season
    taus_season = []
    for acf_comp, label in [(acf_u_s, "u"), (acf_v_s, "v")]:
        below = np.where(~np.isnan(acf_comp) & (acf_comp < threshold))[0]
        if len(below) > 0:
            taus_season.append(float(lags_s[below[0]]))
    if taus_season:
        seasonal_taus[season_name] = np.mean(taus_season)

fig_seasonal.update_xaxes(title_text="Lag (hours)", row=2)
fig_seasonal.update_yaxes(title_text="ACF", col=1)
fig_seasonal.update_layout(
    height=500, template="plotly_white",
    title_text=f"Seasonal Surface Current Autocorrelation — lat={LAT}, lon={LON}",
)
fig_seasonal.show()

print("\nSeasonal decorrelation timescales:")
for name, tau in seasonal_taus.items():
    print(f"  {name}: τ ≈ {tau:.0f}h ({tau/24:.1f} days)")


Seasonal decorrelation timescales:
  DJF (winter): τ ≈ 23h (1.0 days)
  MAM (spring): τ ≈ 8h (0.3 days)
  JJA (summer): τ ≈ 5h (0.2 days)
  SON (autumn): τ ≈ 26h (1.1 days)
